# Assignment 01 - Intelligent Document Analyzer
> **Assignment 01**
> **Run cells in order: 1 → 2 → 3 → 4**


In [ ]:

#   Cell 1 — Install packages  (run once, then Cell 2)

import subprocess, sys

packages = [
    "pymupdf",       # PDF parsing  (imports as fitz)
    "python-docx",   # DOCX parsing (imports as docx)
    "nltk",          # tokenization / stopwords
    "scikit-learn",  # TF-IDF, cosine similarity
    "numpy",         # array ops for LexRank
    "gradio",        # web UI
]

subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet"] + packages)
print("All packages installed.")


All packages installed.


In [ ]:

#   Cell 2 — All imports + NLTK data download


#     standard library
import re, math, io, os, json
from dataclasses import dataclass, field
from collections import Counter
from typing import List, Tuple

#    third-party
import fitz                                         # PyMuPDF  — pip: pymupdf
from docx import Document as DocxDocument           # pip: python-docx
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import sent_tokenize, word_tokenize
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import gradio as gr

#    NLTK data
for _pkg in ("punkt", "punkt_tab", "stopwords"):
    nltk.download(_pkg, quiet=True)

_STOP = set(stopwords.words("english"))

print("All imports OK. NLTK data ready.")


All imports OK. NLTK data ready.


In [ ]:

#   Cell 3 — Core logic (no imports needed here)



# A) DATA CLASSES


@dataclass
class Paragraph:
    text: str
    page: int
    is_heading: bool = False
    heading_level: int = 0

@dataclass
class ExtractionResult:
    paragraphs: List[Paragraph] = field(default_factory=list)
    full_text: str = ""
    filename: str = ""
    page_count: int = 0

@dataclass
class HeadingEntry:
    level: int
    text: str
    page: int
    number: str = ""

@dataclass
class SearchResult:
    text: str
    page: int
    score: float
    rank: int



# B) EXTRACTION


def _pattern_level(text):
    """Detect heading level from numbering/caps patterns."""
    if re.match(r"^\d+[\.)][\s]+\w", text) and len(text) < 100: return 1
    if re.match(r"^\d+\.\d+[\.)][\s]+\w", text) and len(text) < 100: return 2
    if re.match(r"^\d+\.\d+\.\d+[\s]+\w", text) and len(text) < 100: return 3
    if text.isupper() and 3 < len(text) < 80: return 1
    return 0


def extract_pdf(file_bytes, filename):
    doc    = fitz.open(stream=file_bytes, filetype="pdf")
    result = ExtractionResult(filename=filename, page_count=len(doc))
    parts  = []
    for page_num, page in enumerate(doc, 1):
        for block in page.get_text("dict")["blocks"]:
            if block["type"] != 0: continue
            for line in block.get("lines", []):
                text = " ".join(s["text"] for s in line["spans"]).strip()
                if not text: continue
                sizes  = [s["size"]  for s in line["spans"]]
                flags  = [s["flags"] for s in line["spans"]]
                avg_sz = sum(sizes) / len(sizes) if sizes else 12
                bold   = any(f & 16 for f in flags)
                lvl = 0
                if avg_sz >= 18 or (bold and avg_sz >= 16): lvl = 1
                elif avg_sz >= 14 or (bold and avg_sz >= 13): lvl = 2
                elif bold and len(text) < 80: lvl = 3
                if lvl == 0: lvl = _pattern_level(text)
                result.paragraphs.append(Paragraph(text, page_num, lvl > 0, lvl))
                parts.append(text)
    result.full_text = "\n".join(parts)
    doc.close()
    return result


def extract_docx(file_bytes, filename):
    doc    = DocxDocument(io.BytesIO(file_bytes))
    result = ExtractionResult(filename=filename, page_count=1)
    parts  = []
    STYLES = {"heading 1":1,"heading 2":2,"heading 3":3,"title":1,"subtitle":2}
    for para in doc.paragraphs:
        text = para.text.strip()
        if not text: continue
        lvl  = STYLES.get((para.style.name or "").lower(), 0)
        if lvl == 0: lvl = _pattern_level(text)
        result.paragraphs.append(Paragraph(text, 1, lvl > 0, lvl))
        parts.append(text)
    result.full_text = "\n".join(parts)
    return result


def extract(file_bytes, filename):
    if filename.lower().endswith(".pdf"):  return extract_pdf(file_bytes, filename)
    if filename.lower().endswith(".docx"): return extract_docx(file_bytes, filename)
    raise ValueError(f"Unsupported file type: {filename}")



# C) SUMMARIZATION — LexRank (pure sklearn, no sumy)


def _word_count(text):
    return len(word_tokenize(text))


def _lexrank(text, ratio):
    """
    Graph-based extractive summarization.
    1. Vectorize sentences with TF-IDF.
    2. Build cosine similarity matrix (= sentence graph edge weights).
    3. Row-normalize into a Markov matrix.
    4. Run PageRank (power iteration).
    5. Pick top-N sentences by score, preserving document order.
    """
    sentences = sent_tokenize(text)
    if len(sentences) < 3:
        return text

    vec = TfidfVectorizer(stop_words="english", sublinear_tf=True)
    mat = vec.fit_transform(sentences).toarray()
    sim = cosine_similarity(mat)

    row_sums = sim.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    sim = sim / row_sums

    d      = 0.85
    scores = np.ones(len(sentences)) / len(sentences)
    for _ in range(100):
        scores = (1 - d) / len(sentences) + d * sim.T @ scores

    n   = max(3, math.ceil(len(sentences) * ratio))
    top = sorted(np.argsort(scores)[-n:])
    return " ".join(sentences[i] for i in top)


def _freq_based(text, ratio):
    """Fallback: score sentences by normalized term frequency."""
    sents  = sent_tokenize(text)
    if not sents: return text
    words  = [w.lower() for w in word_tokenize(text)
              if w.isalpha() and w.lower() not in _STOP]
    freq   = Counter(words)
    mf     = max(freq.values()) if freq else 1
    nf     = {w: f / mf for w, f in freq.items()}
    scores = {i: sum(nf.get(w.lower(), 0) for w in word_tokenize(s) if w.isalpha())
              for i, s in enumerate(sents)}
    n   = max(3, math.ceil(len(sents) * ratio))
    top = sorted(sorted(scores, key=scores.get, reverse=True)[:n])
    return " ".join(sents[i] for i in top)


def summarize(text, ratio=0.15):
    text  = re.sub(r"\s+", " ", text).strip()
    ratio = max(ratio, 0.10)   # assignment requires >= 10%
    orig  = _word_count(text)

    try:
        summary = _lexrank(text, ratio)
        method  = "LexRank (TF-IDF PageRank)"
    except Exception:
        summary = _freq_based(text, ratio)
        method  = "Frequency-based TF scoring"

    sw = _word_count(summary)
    if orig > 0 and sw / orig < 0.10:          # enforce floor
        summary = _freq_based(text, 0.20)
        sw      = _word_count(summary)
        method += " + expanded"

    return {
        "summary":        summary,
        "method":         method,
        "original_words": orig,
        "summary_words":  sw,
        "ratio_pct":      round(sw / orig * 100, 1) if orig else 0,
    }



# D) HEADING EXTRACTION & TABLE OF CONTENTS


def _strip_numbering(text):
    text = re.sub(r"^[\d\.]+\s*", "", text)
    return re.sub(r"^[A-Z]\.\s*", "", text).strip()


def extract_headings(paragraphs):
    entries  = []
    counters = [0, 0, 0, 0]
    for p in paragraphs:
        if not p.is_heading: continue
        lvl  = min(p.heading_level, 4)
        text = _strip_numbering(p.text)
        if not text: continue
        counters[lvl - 1] += 1
        for i in range(lvl, 4): counters[i] = 0
        num = ".".join(str(c) for c in counters[:lvl] if c > 0)
        entries.append(HeadingEntry(lvl, text, p.page, num))
    return entries


def toc_text(headings):
    if not headings:
        return "(No headings detected)"
    lines = []
    for h in headings:
        indent = "  " * (h.level - 1)
        pg     = f"  (p.{h.page})" if h.page > 1 else ""
        lines.append(f"{indent}{h.number}  {h.text}{pg}")
    return "\n".join(lines)



# E) SEMANTIC SEARCH — TF-IDF cosine similarity


def _split_into_sentences(paragraphs, min_len=30):
    chunks = []
    for p in paragraphs:
        if p.is_heading: continue
        for s in sent_tokenize(p.text):
            if len(s) >= min_len:
                chunks.append((s, p.page))
    return chunks


class TFIDFSearcher:
    def __init__(self, paragraphs):
        data         = _split_into_sentences(paragraphs)
        self._texts  = [d[0] for d in data]
        self._pages  = [d[1] for d in data]
        self._vec    = TfidfVectorizer(stop_words="english", ngram_range=(1, 2),
                                       min_df=1, sublinear_tf=True)
        self._mat    = self._vec.fit_transform(self._texts) if self._texts else None

    def search(self, query, top_k=5):
        if self._mat is None or not query.strip(): return []
        sims    = cosine_similarity(self._vec.transform([query]), self._mat).flatten()
        indices = np.argsort(sims)[::-1][:top_k]
        results = []
        for rank, i in enumerate(indices, 1):
            sc = float(sims[i])
            if sc < 1e-6: break
            results.append(SearchResult(self._texts[i], self._pages[i],
                                        round(sc * 100, 1), rank))
        return results


print("All logic defined — ready for Cell 4 (UI).")


All logic defined — ready for Cell 4 (UI).


In [ ]:

#  Cell 4 — Launch Gradio UI


_state = {}   # persists extraction + searcher between button clicks


def _read_upload(filepath):
    """Read Gradio temp file → (bytes, filename)."""
    with open(filepath, "rb") as f:
        data = f.read()
    return data, os.path.basename(filepath)


def _ensure_loaded(filepath):
    """Extract document on first call or when a new file is uploaded."""
    if not filepath:
        return None, "Please upload a PDF or DOCX file first."
    data, name = _read_upload(filepath)
    if _state.get("filename") != name:
        result              = extract(data, name)
        _state["filename"]  = name
        _state["result"]    = result
        _state["searcher"]  = TFIDFSearcher(result.paragraphs)
    return _state["result"], None


#    button handlers

def do_summarize(filepath, ratio_pct):
    result, err = _ensure_loaded(filepath)
    if err: return err, ""
    sd    = summarize(result.full_text, ratio=ratio_pct / 100)
    stats = (
        f"**Method:** {sd['method']}  \n"
        f"**Original:** {sd['original_words']:,} words  \n"
        f"**Summary:** {sd['summary_words']:,} words  \n"
        f"**Ratio achieved:** {sd['ratio_pct']}%"
    )
    return sd["summary"], stats


def do_toc(filepath):
    result, err = _ensure_loaded(filepath)
    if err: return err
    return toc_text(extract_headings(result.paragraphs))


def do_search(filepath, query, top_k):
    result, err = _ensure_loaded(filepath)
    if err: return err
    if not query.strip(): return "Please enter a search query."
    hits = _state["searcher"].search(query, top_k=int(top_k))
    if not hits: return "No relevant passages found. Try a different query."
    parts = []
    for r in hits:
        pg = f"  ·  page {r.page}" if r.page > 1 else ""
        parts.append(f"**#{r.rank}  Score: {r.score}%{pg}**\n\n{r.text}")
    return "\n\n---\n\n".join(parts)


#    UI layout

with gr.Blocks(title="DocMind", theme=gr.themes.Soft()) as demo:

    gr.Markdown("""
# 📄 DocMind — Intelligent Document Analyzer
**Assignment 01** · Classical NLP · No LLM APIs · Python 3.12 compatible
""")

    file_input = gr.File(
        label="Upload PDF or DOCX",
        file_types=[".pdf", ".docx"],
    )

    with gr.Tabs():

        with gr.TabItem("📋  Summarization"):
            ratio_slider = gr.Slider(10, 40, value=15, step=5,
                                     label="Summary length (% of original, minimum 10%)")
            summ_btn = gr.Button("Generate Summary", variant="primary")
            summ_out = gr.Textbox(label="Summary", lines=12, show_copy_button=True)
            stats_md = gr.Markdown()
            summ_btn.click(fn=do_summarize,
                           inputs=[file_input, ratio_slider],
                           outputs=[summ_out, stats_md])

        with gr.TabItem("📑  Table of Contents"):
            toc_btn = gr.Button("Extract Headings", variant="primary")
            toc_out = gr.Textbox(label="Table of Contents", lines=20,
                                 show_copy_button=True,
                                 placeholder="Upload a file then click Extract Headings")
            toc_btn.click(fn=do_toc, inputs=[file_input], outputs=[toc_out])

        with gr.TabItem("🔍  Semantic Search"):
            with gr.Row():
                query_in = gr.Textbox(
                    label="Keyword, phrase, or question",
                    placeholder="e.g.  preprocessing steps  /  what are the objectives?",
                    scale=4,
                )
                topk_in = gr.Slider(3, 10, value=5, step=1, label="Results", scale=1)
            search_btn = gr.Button("Search", variant="primary")
            search_out = gr.Markdown()
            search_btn.click(fn=do_search,
                             inputs=[file_input, query_in, topk_in],
                             outputs=[search_out])

    gr.Markdown("---\n*DocMind · sumy-free · Python 3.12 · Assignment 01*")


# share=True generates a public gradio.live URL inside Colab
demo.launch(share=True)


/tmp/ipykernel_10530/807080294.py:63: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="DocMind", theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6a2f3bd78180aa4e2d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
